# Model 4 — Twin Basin, Vertically Variable Density

**Gravity inversion** for a twin sedimentary basin with a depth-dependent density contrast:

$$\Delta\rho(z) = -300 + 0.05z \quad [\text{kg/m}^3]$$


## Imports and core dependencies
This cell loads numerical, interpolation, optimisation, and gravity‑forward libraries used throughout the inversion workflow.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.ticker import MultipleLocator
from scipy.interpolate import RectBivariateSpline, griddata
from scipy.optimize import differential_evolution, minimize
from gravity3d_variable_density1 import compute_gravity
import time

print('All imports successful.')

## Spatial grid definition
Here the 3D model domain (15×15 km, 2 km depth) and its cell edges/centres are defined, along with the 2D observation grid at the surface.

In [ ]:
# ── Domain ─────────────────────────────────────────────────────────────────
Lx, Ly    = 15_000.0, 15_000.0
max_depth = 2_000.0

nx, ny, nz = 30, 30, 25

xe = np.linspace(0, Lx, nx + 1);  ye = np.linspace(0, Ly, ny + 1)
ze = np.linspace(0, max_depth, nz + 1)
xc = 0.5 * (xe[:-1] + xe[1:]);  yc = 0.5 * (ye[:-1] + ye[1:])
zc = 0.5 * (ze[:-1] + ze[1:])
X2d, Y2d = np.meshgrid(xc, yc, indexing='ij')

print(f'Full grid : {nx}x{ny}x{nz}')

## Depth‑dependent density contrast law
This cell specifies the linear density contrast function Δρ(z) = drho0 + α z and prints example values at shallow and deep levels.

In [ ]:
# ── Depth-dependent density contrast ───────────────────────────────────────
drho0 = -300.0
alpha =   0.05

def density_contrast_at_depth(z_val):
    return drho0 + alpha * z_val

print(f'Density law : drho(z) = {drho0} + {alpha}*z  kg/m3')
print(f'  at z=0    : {drho0:.0f} kg/m3')
print(f'  at z=2000 : {drho0 + alpha*2000:.0f} kg/m3')

## Construction of true twin‑basin geometry
Two parabolic basins with different centres, radii, and maximum depths are built and combined into a single synthetic “true” basin model.

In [ ]:
# ── True twin-basin geometry ────────────────────────────────────────────────
max_depth1, max_depth2 = 1600.0, 1100.0
x1, y1, Rmax1 = Lx * 0.30, Ly / 2, 3900.0
x2, y2, Rmax2 = Lx * 0.65, Ly / 2, 4500.0

R1 = np.sqrt((X2d - x1)**2 + (Y2d - y1)**2)
R2 = np.sqrt((X2d - x2)**2 + (Y2d - y2)**2)
b1 = max_depth1 * (1 - (R1 / Rmax1)**2);  b1[R1 > Rmax1] = 0.0
b2 = max_depth2 * (1 - (R2 / Rmax2)**2);  b2[R2 > Rmax2] = 0.0
basin_true = np.maximum(b1, b2)

print(f'Basin 1 : centre=({x1/1e3:.1f}, {y1/1e3:.1f}) km  '
      f'Rmax={Rmax1/1e3} km  depth={max_depth1:.0f} m')
print(f'Basin 2 : centre=({x2/1e3:.1f}, {y2/1e3:.1f}) km  '
      f'Rmax={Rmax2/1e3} km  depth={max_depth2:.0f} m')
print(f'True max depth : {basin_true.max():.1f} m')

## Building the 3D density contrast volume
Given a depth surface, this function fills a 3D array with depth‑dependent density contrast in each cell column using the Δρ(z) law.

In [ ]:
# ── Density volume builder ──────────────────────────────────────────────────
def build_rho(depth_surface):
    """Returns rho_contrast[nx, ny, nz] with depth-dependent contrast."""
    nx_l = depth_surface.shape[0]
    rho = np.zeros((nx_l, ny, nz))
    for k in range(nz):
        drho_k = density_contrast_at_depth(zc[k])
        rho[:, :, k][zc[k] < depth_surface] = drho_k
    return rho

print('build_rho defined.')

## Forward gravity calculation and noisy observations
This cell computes gravity from the true density model over the observation grid, then adds Gaussian noise to generate synthetic observed data.

In [ ]:
# ── Forward gravity (full grid) ─────────────────────────────────────────────
Xobs = X2d.copy()
Yobs = Y2d.copy()
Zobs = np.zeros_like(X2d)

print('Computing full-resolution forward gravity ...')
t0 = time.time()
rho_true = build_rho(basin_true)
gz_clean = compute_gravity(Xobs, Yobs, Zobs, xe, ye, ze, rho_true)

np.random.seed(42)
gz_obs = gz_clean + np.random.normal(0.0, 1.0, gz_clean.shape)
#gz_obs = gz_clean
print(f'Done in {time.time()-t0:.1f}s  |  '
      f'gz: {gz_clean.min():.2f} -> {gz_clean.max():.2f} mGal')
print(f'Obs pts : {nx}x{ny} = {nx*ny}')

## B‑spline parametrisation with 16×16 control grid
The basin surface is parameterised via a 16×16 control‑point grid and interpolated to the full grid using bicubic splines, with depth bounds [0, 2000] m.

In [ ]:
# ── B-spline parametrisation ────────────────────────────────────────────────
# IMPROVEMENT: Increase control points to 16x16 for finer basin wall resolution.
# 12x12=144 params was still too coarse to resolve sharp parabolic basin edges.
n_cx, n_cy = 10, 10
x_ctrl     = np.linspace(0, Lx, n_cx)
y_ctrl     = np.linspace(0, Ly, n_cy)
dlo, dhi   = 0.0, 2000.0

def bspline(params, xout, yout):
    spl = RectBivariateSpline(x_ctrl, y_ctrl,
                              params.reshape(n_cx, n_cy), kx=3, ky=3)
    return np.clip(spl(xout, yout, grid=True), dlo, dhi)

print(f'Control grid: {n_cx}x{n_cy} = {n_cx*n_cy} parameters')


## Depth‑sensitivity weighting (conceptual FIX 2)
An empirical depth‑sensitivity profile is computed to reflect reduced gravity sensitivity at depth under variable density; it can be used to weight misfit or regularisation.

In [ ]:
# ── FIX 2: Depth-sensitivity weighting matrix ──────────────────────────────
# With variable density drho(z) = drho0 + alpha*z, the gravity kernel
# attenuates quickly with depth. We compute a depth-sensitivity weight
# w(z) ~ |drho(z)| * dz / z^1.5  (empirical proxy for vertical resolution)
# and use it to build a column-integrated sensitivity weight per surface cell.
# This weight rescales the data misfit so deeper basins are not penalised more.
dz_val = ze[1] - ze[0]
_sens_col = np.zeros(nz)
for k in range(nz):
    _sens_col[k] = abs(density_contrast_at_depth(zc[k])) * dz_val / max(zc[k], 50.0)**1.0
# Normalise: weight per metre of basin depth at each cell column
# We will use this in the regularised misfit below.

## 2‑D Laplacian smoothness operator
This function builds a finite‑difference Laplacian matrix on the control grid to penalise roughness in the depth parameters (Tikhonov regularisation).

In [ ]:
# ── FIX 3: Tikhonov smoothness matrix (2-D Laplacian on control grid) ──────
def _build_laplacian_reg(nc):
    """Build 2-D finite-difference Laplacian matrix for nc x nc grid."""
    N  = nc * nc
    L  = np.zeros((N, N))
    for i in range(nc):
        for j in range(nc):
            idx = i * nc + j
            cnt = 0
            if i > 0:      L[idx, (i-1)*nc+j] = -1; cnt += 1
            if i < nc-1:   L[idx, (i+1)*nc+j] = -1; cnt += 1
            if j > 0:      L[idx, i*nc+(j-1)] = -1; cnt += 1
            if j < nc-1:   L[idx, i*nc+(j+1)] = -1; cnt += 1
            L[idx, idx] = cnt
    return L

_L = _build_laplacian_reg(n_cx)   # n_cx == n_cy assumed

## Regularised misfit: data, smoothness, and depth‑bias terms
The objective combines variance‑normalised gravity misfit, Laplacian smoothness (λ_s), and a depth‑bias term (λ_d) toward a reference twin‑basin surface; parameter bounds are also set here.

In [ ]:
# ── Regularised misfit with depth-weighting + smoothness ────────────────────
# IMPROVEMENT: Reduce lambda_s from 5e-4 → 1e-4 to allow sharper basin walls.
# Activate lambda_d=1e-4 to counteract gravity-depth ambiguity (pushes inversion
# toward deeper solutions where density is weaker, i.e. penalises shallow bias).
#
# lambda_s  : smoothness (Tikhonov) weight   — penalises rough surfaces
# lambda_d  : depth bias weight — mild preference for solutions near true depths
lambda_s = 5e-3   # smoothness — raised from 1e-4
lambda_d = 0.3    # depth prior — raised from 1e-4 (was effectively zero)

_gz_var = float(np.var(gz_obs))   # data variance for normalisation

# Pre-compute reference depths on control grid for depth-bias term
_X_ctrl, _Y_ctrl = np.meshgrid(x_ctrl, y_ctrl, indexing='ij')
_R1_ctrl = np.sqrt((_X_ctrl - x1)**2 + (_Y_ctrl - y1)**2)
_R2_ctrl = np.sqrt((_X_ctrl - x2)**2 + (_Y_ctrl - y2)**2)
_b1_ref  = max_depth1 * np.clip(1 - (_R1_ctrl / Rmax1)**2, 0, None)
_b2_ref  = max_depth2 * np.clip(1 - (_R2_ctrl / Rmax2)**2, 0, None)
_depth_ref = np.maximum(_b1_ref, _b2_ref).ravel()  # reference depth surface

def misfit(params):
    surf    = bspline(params, xc, yc)
    gz_pred = compute_gravity(Xobs, Yobs, Zobs, xe, ye, ze, build_rho(surf))
    # Normalised data misfit
    data_term   = float(np.mean((gz_pred - gz_obs)**2)) / _gz_var
    # Smoothness regularisation on control-point values
    p = params / dhi
    smooth_term = float(p @ _L @ p) / (n_cx * n_cy)
    # Depth-bias: penalise deviation from reference depth surface
    depth_bias  = float(np.mean(((params - _depth_ref) / dhi)**2))
    return data_term + lambda_s * smooth_term + lambda_d * depth_bias

bounds = [(dlo, dhi)] * (n_cx * n_cy)
print(f'Parameters : {n_cx}x{n_cy} = {n_cx*n_cy}  |  bounds: [{dlo:.0f}, {dhi:.0f}] m')
print(f'Regularisation: lambda_s={lambda_s}  lambda_d={lambda_d}')


## Differential evolution callback for logging
This callback records best misfit, candidate value, elapsed time, and convergence metric at each DE iteration and keeps a history of best parameter vectors.

In [ ]:
# ── Differential Evolution callback ────────────────────────────────────────
_iter = [0]; _t0 = [time.time()]; _best = [np.inf]
_hist_x = []; _hist_f = []

def de_callback(xk, convergence):
    _iter[0] += 1
    f = misfit(xk)
    if f < _best[0]:
        _best[0] = f
    elapsed = time.time() - _t0[0]
    _hist_x.append(xk.copy())
    _hist_f.append(_best[0])
    print(f'  DE iter {_iter[0]:>4d} | best misfit = {_best[0]:.6f} | '
          f'candidate = {f:.6f} | elapsed = {elapsed:.1f}s | '
          f'convergence = {convergence:.4f}')

print('Callback ready.')

## Stage 1: Global search via differential evolution
Differential evolution (strategy=randtobest1bin) explores the full parameter space using the regularised misfit to find a good global candidate for the basin surface.

In [ ]:
# Stage 1 Differential Evolution
print("-" * 70)
print(f"STAGE 1 Differential Evolution full {nx}x{ny}x{nz} grid")
print("strategy=randtobest1bin, popsize=20, maxiter=3000")
print(f"control pts {n_cx}x{n_cy}={n_cx*n_cy}  lambdas={lambda_s}")
print("-" * 70)

iter0 = 0
t00 = time.time()
best0 = np.inf
_hist_x.clear()
_hist_f.clear()

popsize  = 15
n_params = n_cx * n_cy

# Warm-start: half the population around the depth prior, half from LHC.
# Noise sigma = depth_max * 0.40 (1200 m) so that DE mutation steps
# (~F * std ≈ 0.5 * 1200 = 600 m, 15% of [0,4000] range) are large
# enough to explore the landscape. The previous sigma=0.15 gave steps of
# only ~180 m (4.5% of range) — too tight for the weaker VD gravity signal.
rng = np.random.default_rng(42)
pop_prior = np.clip(
    _depth_ref[np.newaxis, :] +
    rng.normal(0, max_depth * 0.70, (popsize * n_params // 2, n_params)),
    dlo, dhi)
from scipy.stats import qmc
sampler  = qmc.LatinHypercube(d=n_params, seed=42)
pop_lhc  = qmc.scale(sampler.random(popsize * n_params - len(pop_prior)),
                     dlo, dhi)
init_pop = np.vstack([pop_prior, pop_lhc])

de = differential_evolution(
    misfit,
    bounds=bounds,
    strategy="randtobest1bin",
    maxiter=600,
    popsize=popsize,
    tol=1e-9,
    mutation=(0.5, 1.0),    # raised lower bound: 0.4→0.5 for better exploration
    recombination=0.85,     # slightly lower: 0.90→0.85 increases trial diversity
    polish=False,
    seed=42,
    disp=False,
    callback=de_callback,
    init=init_pop,
    workers=1,
)

print(f"\nDE finished in {time.time()-_t0[0]:.1f}s")
print(f"converged  = {de.success}")
print(f"best misfit = {de.fun:.8f}")

## Stage 2: Regularised L‑BFGS‑B polishing
Starting from the DE solution, L‑BFGS‑B minimises the same regularised misfit to efficiently refine the control‑point depths under the same penalties.

In [ ]:
# ── Stage 2: L-BFGS-B local polish (regularised) ────────────────────────────
print('=' * 70)
print('  STAGE 2 : L-BFGS-B  (regularised polish)')
print('=' * 70)
t_lb = time.time()

lb = minimize(
    misfit, x0=de.x, method='L-BFGS-B', bounds=bounds,
    options={'maxiter': 5000, 'ftol': 1e-15, 'gtol': 1e-10, 'disp': True})

print(f'\nL-BFGS-B finished in {time.time()-t_lb:.1f}s')
print(f'  converged  : {lb.success}')
print(f'  DE  misfit : {de.fun:.8f}')
print(f'  LB  misfit : {lb.fun:.8f}')
print(f'  Improvement: {de.fun - lb.fun:.8f}')


## Stage 3a & 3b: Data‑only L‑BFGS‑B polishes
Two successive L‑BFGS‑B runs minimise a data‑only, variance‑normalised misfit (no regularisation) to remove regularisation bias and sharpen basin edges; the best data‑misfit model is retained.

In [ ]:
# Stage 3: Final regularised L-BFGS-B polish with ultra-tight tolerances
# FIX 7: Replaced data-only polishing with regularised polish.
#        Data-only misfit (Stage 3 original) drops the depth prior which
#        was the only term constraining depth uniqueness. Without it the
#        optimizer fits observational noise at the cost of depth accuracy —
#        RMS gravity residual falls but RMS depth error rises. This is the
#        classic underdetermined inversion trade-off.
# FIX 8: Model selection uses the regularised (full) misfit, not data-only.
print("=" * 65)
print("  STAGE 3 : Ultra-tight regularised L-BFGS-B")
print("=" * 65)
t_lb2 = time.time()
lb2_result = minimize(
    misfit, x0=lb.x, method='L-BFGS-B', bounds=bounds,
    options={'maxiter': 5000, 'ftol': 1e-16, 'gtol': 1e-11, 'disp': True},
)
print(f'Stage-3 finished in {time.time()-t_lb2:.1f}s  misfit={lb2_result.fun:.8f}')

# Select best by regularised misfit (the objective that encodes depth knowledge)
candidates = [(de.x,         misfit(de.x)),
              (lb.x,  lb.fun),
              (lb2_result.x, lb2_result.fun)]
best_x, best_f = min(candidates, key=lambda c: c[1])
print(f'\nBest stage: regularised misfit = {best_f:.8f}')

## Quantitative assessment of recovered basin
This cell computes RMS gravity residual, RMS depth error, and compares true vs recovered maximum depth to measure inversion accuracy.

In [ ]:
# ── Recovered surface & residuals ──────────────────────────────────────────
recovered = bspline(best_x, xc, yc)

gz_rec   = compute_gravity(Xobs, Yobs, Zobs, xe, ye, ze, build_rho(recovered))
residual = gz_rec - gz_obs
rms_grav  = float(np.sqrt(np.mean(residual**2)))
rms_depth = float(np.sqrt(np.mean((recovered - basin_true)**2)))

print(f'RMS gravity residual : {rms_grav:.4f} mGal')
print(f'RMS depth error      : {rms_depth:.1f} m')
print(f'True max depth       : {basin_true.max():.1f} m')
print(f'Recovered max depth  : {recovered.max():.1f} m')

## Mean density contrast of true and recovered basins
The vertically averaged density contrast is computed for both true and recovered surfaces to compare effective density structures under the variable density law.

In [ ]:
# ── Mean density contrast maps ──────────────────────────────────────────────
def mean_density_contrast(depth_surface):
    dz_val = ze[1] - ze[0]
    nx_l, ny_l = depth_surface.shape
    num = np.zeros((nx_l, ny_l))
    for k in range(nz):
        drho_k = density_contrast_at_depth(zc[k])
        cell_in = np.minimum(dz_val,
                             np.maximum(0.0, depth_surface - (zc[k] - dz_val / 2)))
        num += drho_k * cell_in
    with np.errstate(invalid='ignore', divide='ignore'):
        return np.where(depth_surface > 0, num / depth_surface, 0.0)

drho_true_mean = mean_density_contrast(basin_true)
drho_rec_mean  = mean_density_contrast(recovered)

xkm = xc / 1e3;  ykm = yc / 1e3
XX, YY = np.meshgrid(xkm, ykm, indexing='ij')

print('Mean density contrast maps computed.')

## Plot (a): Gravity anomaly maps

In [ ]:
# ── Plot (a): Gravity anomaly maps ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
fig.suptitle(
    '(a)  Gravity Anomaly Maps — Model 4 (Twin Basin, Vertically Variable Density)',
    fontsize=13, fontweight='bold')

gzlo = float(min(gz_obs.min(), gz_rec.min()))
gzhi = float(max(gz_obs.max(), gz_rec.max()))
vr   = float(np.max(np.abs([residual.min(), residual.max()])))

for ax, dat, ttl, cmp, (vlo, vhi) in zip(
        axes,
        [gz_obs,   gz_rec,   residual],
        ['Observed Anomaly (mGal)', 'Recovered Anomaly (mGal)', 'Residual Anomaly (mGal)'],
        ['jet',    'jet',    'RdBu_r'],
        [(gzlo, gzhi), (gzlo, gzhi), (-vr, vr)]):

    cf = ax.contourf(XX, YY, dat,
                     levels=np.linspace(vlo, vhi, 200),
                     cmap=cmp, extend='both')
    ax.axhline(7.5, color='white', linestyle='--', linewidth=1.5,
               label='y = 7.5 km')
    ax.set_title(ttl, fontweight='bold', fontsize=11)
    ax.set_xlabel('x (km)', fontweight='bold', fontsize=11)
    ax.set_ylabel('y (km)', fontweight='bold', fontsize=11)
    ax.set_xlim(0, 15);  ax.set_ylim(0, 15)
    ax.xaxis.set_major_locator(MultipleLocator(5))
    ax.yaxis.set_major_locator(MultipleLocator(5))
    ax.grid(True, linestyle='--', linewidth=0.4, alpha=0.5, color='k')
    cb = plt.colorbar(cf, ax=ax, pad=0.02)
    cb.locator   = mticker.MaxNLocator(nbins=6)
    cb.formatter = mticker.FormatStrFormatter('%.0f')
    cb.update_ticks()
    label_txt = 'gz (mGal)' if 'Residual' not in ttl else 'Residual gz (mGal)'
    cb.set_label(label_txt, fontsize=10)

plt.tight_layout()
plt.savefig('model4_a_gravity.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: model4_a_gravity.png')

## Plot (b): Basin depth & density structure

In [ ]:
# ── Plot (b): Basin depth & density structure ───────────────────────────────
vmax_d   = float(max(basin_true.max(), recovered.max()))
levels_d = np.linspace(0, vmax_d, 60)

vmin_r   = min(drho_true_mean.min(), drho_rec_mean.min())
vmax_r   = max(drho_true_mean.max(), drho_rec_mean.max())
levels_r = np.linspace(vmin_r, vmax_r, 60)

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle(
    '(b)  Basin Depth & Density Structure — Model 4 (Twin Basin, Vertically Variable Density)',
    fontsize=13, fontweight='bold')

last_depth_cf = None
for ax, (dat, ttl) in zip(axes[0],
        [(basin_true, 'True Basin Depth (m)'),
         (recovered,  'Recovered Basin Depth (m)')]):
    cf = ax.contourf(XX, YY, dat, levels=levels_d, cmap='jet', extend='max')
    ax.axhline(7.5, color='white', linestyle='--', linewidth=2.0,
               label='y = 7.5 km')
    ax.set_title(ttl, fontweight='bold', fontsize=12)
    ax.set_xlabel('x (km)', fontweight='bold', fontsize=11, labelpad=10)
    ax.set_ylabel('y (km)', fontweight='bold', fontsize=11, labelpad=8)
    ax.set_xlim(0, 15);  ax.set_ylim(0, 15)
    ax.xaxis.set_major_locator(MultipleLocator(5))
    ax.yaxis.set_major_locator(MultipleLocator(5))
    ax.tick_params(axis='x', pad=6)
    ax.grid(True, linestyle='--', linewidth=0.4, alpha=0.5, color='k')
    last_depth_cf = cf

last_rho_cf = None
for ax, (dat, ttl) in zip(axes[1],
        [(drho_true_mean, 'True Mean Density Contrast (kg/m³)'),
         (drho_rec_mean,  'Recovered Mean Density Contrast (kg/m³)')]):
    cf = ax.contourf(XX, YY, dat, levels=levels_r, cmap='seismic_r', extend='both')
    ax.axhline(7.5, color='black', linestyle='--', linewidth=1.4)
    ax.set_title(ttl, fontweight='bold', fontsize=12)
    ax.set_xlabel('x (km)', fontweight='bold', fontsize=11, labelpad=10)
    ax.set_ylabel('y (km)', fontweight='bold', fontsize=11, labelpad=8)
    ax.set_xlim(0, 15);  ax.set_ylim(0, 15)
    ax.xaxis.set_major_locator(MultipleLocator(5))
    ax.yaxis.set_major_locator(MultipleLocator(5))
    ax.tick_params(axis='x', pad=6)
    ax.grid(True, linestyle='--', linewidth=0.4, alpha=0.5, color='k')
    last_rho_cf = cf

fig.subplots_adjust(hspace=0.45, wspace=0.3, right=0.88)

cax_d = fig.add_axes([0.91, 0.54, 0.018, 0.34])
cb_d  = fig.colorbar(last_depth_cf, cax=cax_d, orientation='vertical')
cb_d.set_label('Depth (m)', fontweight='bold', fontsize=11, labelpad=12)
cb_d.ax.tick_params(labelsize=9)

cax_r = fig.add_axes([0.91, 0.10, 0.018, 0.34])
cb_r  = fig.colorbar(last_rho_cf, cax=cax_r, orientation='vertical')
cb_r.set_label('Mean Δρ (kg/m³)', fontweight='bold', fontsize=11, labelpad=12)
cb_r.ax.tick_params(labelsize=9)

plt.savefig('model4_b_depth_density.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: model4_b_depth_density.png')

## Plot (c): Vertical cross-section at Y = 7.5 km

In [ ]:
# ── Plot (c): Vertical cross-section at Y = 7.5 km ─────────────────────────
j = int(np.argmin(np.abs(yc - 7500.0)))
print(f'Cross-section at yc[{j}] = {yc[j]/1e3:.3f} km')

true_depth_slice = basin_true[:, j]
rec_depth_slice  = recovered[:, j]

ix_b1 = int(np.argmin(np.abs(xc - x1)))
drho_z_true = np.array([
    density_contrast_at_depth(zk) if zk < basin_true[ix_b1, j] else 0.0
    for zk in zc])
drho_z_rec = np.array([
    density_contrast_at_depth(zk) if zk < recovered[ix_b1, j] else 0.0
    for zk in zc])

z_line   = np.linspace(0, max_depth, 300)
drho_law = drho0 + alpha * z_line

z_fine  = np.linspace(0, max_depth, 300)
Xg, Zg  = np.meshgrid(xkm, z_fine, indexing='ij')
inside_mask = Zg < rec_depth_slice[:, np.newaxis]
drho_grid   = drho0 + alpha * z_fine
drho_2d     = np.where(inside_mask, drho_grid[np.newaxis, :], np.nan)
vmin_fill = np.nanmin(drho_2d)
vmax_fill = np.nanmax(drho_2d)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6.5))
fig.suptitle(
    '(c)  Vertical Cross-Section at Y = 7.5 km — '
    'Model 4 (Twin Basin, Vertically Variable Density)',
    fontsize=13, fontweight='bold')

pm = ax1.pcolormesh(
    xkm, z_fine, drho_2d.T,
    cmap='coolwarm_r',
    vmin=vmin_fill, vmax=vmax_fill,
    shading='auto', zorder=1)

ax1.plot(xkm, rec_depth_slice,
         color='red',  linewidth=2.7, label='Recovered Basin', zorder=3)
ax1.plot(xkm, true_depth_slice,
         linestyle='--', color='blue', linewidth=2.2, label='True Basin', zorder=4)
ax1.fill_between(xkm, true_depth_slice, rec_depth_slice,
                 where=(rec_depth_slice > true_depth_slice),
                 alpha=0.25, color='orange', label='Over-estimated', zorder=2)
ax1.fill_between(xkm, true_depth_slice, rec_depth_slice,
                 where=(rec_depth_slice < true_depth_slice),
                 alpha=0.25, color='green',  label='Under-estimated', zorder=2)
ax1.axvline(x1 / 1e3, color='blue',  linestyle=':', linewidth=1.3, alpha=0.8,
            label=f'Basin 1 ({x1/1e3:.1f} km)')
ax1.axvline(x2 / 1e3, color='green', linestyle=':', linewidth=1.3, alpha=0.8,
            label=f'Basin 2 ({x2/1e3:.1f} km)')

cb1 = plt.colorbar(pm, ax=ax1, pad=0.02, fraction=0.046)
cb1.set_label('dr inside recovered basin (kg/m3)', fontweight='bold', fontsize=10)
cb1.ax.tick_params(labelsize=9)

ax1.set_title('Basin Geometry along X  (y = 7.5 km)',
              fontweight='bold', fontsize=12)
ax1.set_xlabel('x (km)', fontweight='bold', fontsize=12)
ax1.set_ylabel('Depth (m)', fontweight='bold', fontsize=12)
ax1.set_xlim(0, 15);  ax1.set_ylim(0, max_depth)
ax1.invert_yaxis()
ax1.xaxis.set_major_locator(MultipleLocator(2.5))
ax1.yaxis.set_major_locator(MultipleLocator(400))
for lbl in ax1.get_xticklabels() + ax1.get_yticklabels():
    lbl.set_fontweight('bold')
ax1.tick_params(labelsize=11)
ax1.grid(True, linestyle='--', linewidth=0.5, alpha=0.5, zorder=0)
ax1.legend(prop={'weight': 'bold', 'size': 9}, loc='lower center',
           ncol=3, framealpha=0.85)
ax1.text(0.02, 0.04,
         f'RMS gravity: {rms_grav:.3f} mGal\nRMS depth:   {rms_depth:.1f} m',
         transform=ax1.transAxes, fontsize=10, verticalalignment='bottom',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.85))

ax2.fill_betweenx(zc / 1000.0, drho_z_rec,  0,
                  alpha=0.28, color='red',  label='Recovered fill', step='mid')
ax2.fill_betweenx(zc / 1000.0, drho_z_true, 0,
                  alpha=0.18, color='blue', label='True fill',      step='mid')
ax2.step(drho_z_true, zc / 1000.0,
         where='mid', linestyle='--', color='blue', linewidth=2.2, label='True dr(z)')
ax2.step(drho_z_rec,  zc / 1000.0,
         where='mid', color='red',  linewidth=2.7, label='Recovered dr(z)')
ax2.plot(drho_law, z_line / 1000.0,
         color='gray', linewidth=1.5, linestyle=':', label='dr law (full depth)')
ax2.axvline(0, color='black', linewidth=0.8)

ax2.set_title(
    f'Density-Depth Profile at Basin 1 Centre\n'
    f'(x = {x1/1e3:.1f} km,  y = 7.5 km)',
    fontweight='bold', fontsize=12)
ax2.set_xlabel('Density Contrast dr (kg/m3)', fontweight='bold', fontsize=12)
ax2.set_ylabel('Depth (km)', fontweight='bold', fontsize=12)
ax2.invert_yaxis()
ax2.yaxis.set_major_locator(MultipleLocator(0.5))
for lbl in ax2.get_xticklabels() + ax2.get_yticklabels():
    lbl.set_fontweight('bold')
ax2.tick_params(labelsize=11)
ax2.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
ax2.legend(prop={'weight': 'bold', 'size': 10}, loc='lower right')

plt.tight_layout()
plt.savefig('model4_c_xsection.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: model4_c_xsection.png')

## Final Summary

In [ ]:
# ── Final summary ───────────────────────────────────────────────────────────
print('=' * 70)
print('  FINAL SUMMARY — Model 4  (Twin Basin, Vertically Variable Density)')
print('=' * 70)
print(f'  Density law          : drho(z) = {drho0} + {alpha}*z  kg/m3')
print(f'  Inversion grid       : {nx}x{ny}x{nz}  (full resolution)')
print(f'  Control points       : {n_cx}x{n_cy} = {n_cx*n_cy}  (was 8x8=64)')
print(f'  Regularisation       : lambda_s={lambda_s}  (Tikhonov smoothness)')
print(f'  DE  popsize=20, maxiter=1500  converged={de.success}  misfit={de.fun:.8f}')
print(f'  LB  converged={lb.success}   misfit={lb.fun:.8f}')
print(f'  LB2 converged={lb2_result.success}  misfit={lb2_result.fun:.8f}')
print(f'  Improvement (DE->LB2): {de.fun - lb2_result.fun:.8f}')
print(f'  RMS gravity residual : {rms_grav:.4f} mGal')
print(f'  RMS depth error      : {rms_depth:.1f} m')
print(f'  True max depth       : {basin_true.max():.1f} m')
print(f'  Recovered max depth  : {recovered.max():.1f} m')
print('=' * 70)

## DE Convergence Curve

In [ ]:

# ── Plot (d): DE convergence curve ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogy(range(1, len(_hist_f) + 1), _hist_f,
            '-o', markersize=3, color='steelblue', linewidth=1.8)
ax.set_title('DE Convergence Curve — Model 4 (Twin Basin - Variable Density)',
             fontweight='bold', fontsize=13)
ax.set_xlabel('DE Iteration', fontweight='bold', fontsize=12)
ax.set_ylabel('Best Misfit (normalised MSE)', fontweight='bold', fontsize=12)
ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
for l in ax.get_xticklabels() + ax.get_yticklabels():
    l.set_fontweight('bold')
ax.tick_params(labelsize=11)
plt.tight_layout()
plt.savefig('model4_de_convergence.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: model4_de_convergence.png')

## Cost-function topography in PCA space

In [ ]:
# ── 1. Build ensemble of "acceptable" models and their misfits ───────────────
misfit_array = np.array(_hist_f)
models_array = np.array(_hist_x)          # shape (N_models, n_params)
models_array = models_array.T             # shape (n_params, N_models)

misfit_threshold = np.percentile(misfit_array, 40.0)  # best 40% of models
mask_ok = misfit_array <= misfit_threshold

cost_finall  = misfit_array[mask_ok]     # (N_ok,)
model_finall = models_array[:, mask_ok]  # (n_params, N_ok)

print(f"Accepted models for PCA: {model_finall.shape[1]}")

# ── 2. PCA reduction ──────────────────────────────────────────────────────────
def pca_reduction_py(data):
    mean_vec = np.mean(data, axis=1, keepdims=True)
    data_z   = data - mean_vec
    C = np.cov(data_z)
    Evals, W_col = np.linalg.eigh(C)
    idx     = np.argsort(Evals)[::-1]
    Evalues = Evals[idx]
    W       = W_col[:, idx].T   # rows = eigenvectors
    pc      = W @ data_z
    return pc, Evalues, W, mean_vec

pc, Evalues, W, mean_model = pca_reduction_py(model_finall)

# ── 3. Cost-function topography in the PC1-PC2 plane ─────────────────────────
x = pc[0, :]   # PC1 scores
y = pc[1, :]   # PC2 scores

nxg, nyg = 80, 80
xg = np.linspace(x.min(), x.max(), nxg)
yg = np.linspace(y.min(), y.max(), nyg)
Xg, Yg = np.meshgrid(xg, yg, indexing="ij")

Vq = griddata(points=np.vstack([x, y]).T,
              values=cost_finall,
              xi=(Xg, Yg),
              method="linear")

plt.figure(figsize=(7, 5))
cs   = plt.contourf(Xg, Yg, Vq, levels=12, cmap="jet")
cbar = plt.colorbar(cs)
cbar.set_label("Regularised misfit (dimensionless)")
plt.xlabel("Principal component 1")
plt.ylabel("Principal component 2")
plt.title("Cost-function topography in PCA space (twin basin-variable density) noisy data")

# ── 4. Project best model and true model into PCA space ──────────────────────
# FIX: Use best_x (post all polishing), and Z_basin_true sampled on ctrl grid.
params_best = best_x

from scipy.interpolate import RegularGridInterpolator
_interp_true = RegularGridInterpolator(
    (xc, yc), basin_true, method="linear",
    bounds_error=False, fill_value=0.0)
_X_ctrl_2d, _Y_ctrl_2d = np.meshgrid(x_ctrl, y_ctrl, indexing="ij")
true_on_ctrl = _interp_true(
    np.column_stack([_X_ctrl_2d.ravel(), _Y_ctrl_2d.ravel()])
).reshape(n_cx, n_cy)
true_model = true_on_ctrl.ravel()

mean_flat     = mean_model.ravel()
best_centered = params_best - mean_flat
true_centered = true_model  - mean_flat

loc_best = W @ best_centered
loc_true = W @ true_centered

plt.plot(loc_best[0], loc_best[1], "r^", markersize=10, label="Best model (post L-BFGS-B)")
plt.plot(loc_true[0], loc_true[1], "gv", markersize=10, label="True model")
plt.legend(loc="best")
plt.tight_layout()
plt.savefig('model4_pca_noisy.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Best model  PC1={loc_best[0]:.4f}  PC2={loc_best[1]:.4f}")
print(f"True model  PC1={loc_true[0]:.4f}  PC2={loc_true[1]:.4f}")
dist = np.sqrt((loc_best[0]-loc_true[0])**2 + (loc_best[1]-loc_true[1])**2)
print(f"PC-space distance: {dist:.4f}")
